In [7]:
import pandas as pd
import numpy as np
from tqdm import tqdm_pandas
from tqdm.notebook import tqdm
from transformers import BertModel, BertTokenizerFast, Trainer, TrainingArguments, AutoModelForSequenceClassification, AutoTokenizer, BertForSequenceClassification, BertTokenizer
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support,  roc_auc_score, fbeta_score

import warnings
warnings.filterwarnings("ignore")


# hyperparameters
batch_size = 16
epochs = 3
learning_rate = 2e-5


model_name = 'onlplab/alephbert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name , num_labels=2)

checkpoint_path = "models/model_weights_progressive_curiculum_learning_conv_messages.pth"

# Load state dict
state_dict = torch.load(checkpoint_path, map_location="cpu")  # or "cuda" if on GPU

# Load weights into model
bert_model.load_state_dict(state_dict)

# Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at onlplab/alephbert-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(52000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [8]:
conv_info_path = 'trainDatasets/conv_info.csv'
conv_info_df = pd.read_csv(conv_info_path)
conv_info_df['engagement_id'] = conv_info_df['engagement_id'].astype(str)
conv_info_df = conv_info_df.rename(columns={'gsr': 'label'})
train_conv_df, test_conv_df = train_test_split(conv_info_df, test_size=0.2, stratify=conv_info_df['label'],random_state=42)

In [33]:
all_messages_path = 'trainDatasets/messages_anonymized.csv'

all_messages_df = pd.read_csv(all_messages_path)

all_messages_df['engagement_id'] = all_messages_df['engagement_id'].astype(str)
all_messages_df = all_messages_df[all_messages_df['anonymized'].notna()]
all_messages_df['name'] = all_messages_df['name'].fillna('-')
all_messages_df = all_messages_df[all_messages_df["seeker"]]

#split to train and test base on conversation split and concat messages
train_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(train_conv_df["engagement_id"])]
train_all_messages_df = train_all_messages_df.merge(train_conv_df , on="engagement_id")
train_all_messages_df = train_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()

test_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(test_conv_df["engagement_id"])]
test_all_messages_df = test_all_messages_df.merge(test_conv_df , on="engagement_id")
test_all_messages_df = test_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()


In [14]:
def calc_matrics(labels, preds, pred_probs):
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    #roc_auc = roc_auc_score(labels, pred_probs)
    f2 = fbeta_score(labels, preds, beta=2)
    
    print(f'Accuracy: {accuracy}')
    print(f'Precision: {precision}')
    print(f'Recall: {recall}')
    print(f'F1: {f1}')
    #print(f'ROC-AUC: {roc_auc}')
    print(f'F2: {f2}')

In [34]:
bert_model.eval()

def sliding_window_inference(
    conversation,
    tokenizer,
    model,
    max_length=512,
    stride=256,
    device='cuda' if torch.cuda.is_available() else 'cpu'
):
    """
    Performs inference on a conversation using sliding window if needed.

    Args:
        conversation (str): The full conversation text.
        tokenizer: HuggingFace tokenizer.
        model: Trained HuggingFace classification model.
        max_length (int): Max tokens per chunk.
        stride (int): Stride for overlapping chunks.
        device (str): 'cuda' or 'cpu'.

    Returns:
        List[Tensor]: Softmax probabilities per chunk (or one if short).
    """
    model.eval()
    model.to(device)

    # Tokenize without truncation to get full length
    tokens = tokenizer(conversation, return_tensors='pt', truncation=False, add_special_tokens=False)
    input_ids = tokens['input_ids'][0]
    total_len = input_ids.shape[0]

    predictions = []

    # Case 1: The whole conversation fits in one chunk
    if total_len <= max_length:
        padded_input_ids = torch.cat([
            input_ids,
            torch.full((max_length - total_len,), tokenizer.pad_token_id)
        ])
        attention_mask = (padded_input_ids != tokenizer.pad_token_id).long()

        with torch.no_grad():
            outputs = model(
                input_ids=padded_input_ids.unsqueeze(0).to(device),
                attention_mask=attention_mask.unsqueeze(0).to(device)
            )
            logits = outputs.logits.squeeze(0).cpu()
            predictions.append(torch.softmax(logits, dim=-1))

    # Case 2: Conversation is longer than max_length → sliding window
    else:
        for start_idx in range(0, total_len, stride):
            end_idx = start_idx + max_length
            chunk_ids = input_ids[start_idx:end_idx]

            # Pad if shorter than max_length
            if chunk_ids.shape[0] < max_length:
                padding_len = max_length - chunk_ids.shape[0]
                chunk_ids = torch.cat([
                    chunk_ids,
                    torch.full((padding_len,), tokenizer.pad_token_id)
                ])

            attention_mask = (chunk_ids != tokenizer.pad_token_id).long()

            with torch.no_grad():
                outputs = model(
                    input_ids=chunk_ids.unsqueeze(0).to(device),
                    attention_mask=attention_mask.unsqueeze(0).to(device)
                )
                logits = outputs.logits.squeeze(0).cpu()
                predictions.append(torch.softmax(logits, dim=-1))

    return predictions


In [37]:
labels = []
preds = []
pred_probs = []


for index, row in test_all_messages_df.iterrows():
    conv = row["anonymized"]
    predictions = sliding_window_inference(conversation = conv, tokenizer = tokenizer, model = bert_model)
    sucidal_preds = [x[1] for x in predictions]
    avg_pred = max(sucidal_preds).item()
    
    labels.append(row["label"])
    preds.append(int(avg_pred > 0.5))
    pred_probs.append([1 - avg_pred, avg_pred])
    
calc_matrics(labels, preds, pred_probs)

Accuracy: 0.8491814122705473
Precision: 0.5537918871252204
Recall: 0.8602739726027397
F1: 0.6738197424892703
F2: 0.7745436605821411


In [38]:
tokens = tokenizer(conv, return_tensors='pt', truncation=False, add_special_tokens=False)

In [ ]:
lens = []
for conv in train_all_messages_df["anonymized"]:
    tokens = tokenizer(conv, return_tensors='pt', truncation=False, add_special_tokens=False)
    lens.append(tokens["input_ids"].shape[1])


In [85]:
len([x for x in lens if 300 < x < 400])

1494

In [77]:
sum(lens) / len(lens)

767.2915426177087